# 02 · Labelling: correlation-tensor ranks and the 3-tangle
**Paper:** Sec. II B, Table II; Sec. V B, Table VIII  
Correlation tensor $t_{ijk}=\mathrm{Tr}[\rho\,\sigma_i\otimes\sigma_j\otimes\sigma_k]$; ranks of its three 3×9 unfoldings $\{R(T_{123}),R(T_{213}),R(T_{312})\}$ identify the SLOCC class; the 3-tangle $\tau=4\rho_{00}\rho_{77}$ (canonical form) separates GHZ from W.  
**Code:** `ann_entanglement.measures`

In [1]:
import os, sys
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
sys.path.insert(0, os.path.abspath("../src"))  # not needed after `pip install -e .`
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.precision", 3)

In [2]:
from ann_entanglement import states, measures, dataset
rng = np.random.default_rng(1)
rows = []
for cls in states.CLASS_NAMES:
    for _ in range(200):
        rho = states.random_canonical_state(cls, rng)
        r = measures.correlation_tensor_ranks(rho, tol=1e-10)
        rows.append({'class': cls, 'ranks': r, 'tensor_class': measures.ranks_to_class(r), 'tau': measures.three_tangle_canonical(rho)})
sim = pd.DataFrame(rows)
sim.groupby('class').agg(ranks=('ranks', lambda s: sorted(set(s))), tensor_class=('tensor_class', lambda s: sorted(set(s))), tau_min=('tau', 'min'), tau_max=('tau', 'max'))

,ranks,tensor_class,tau_min,tau_max
class,,,,
BS1,"[(1, 3, 3)]",[BS1],0.000e+00,0.000
BS2,"[(3, 1, 3)]",[BS2],0.000e+00,0.000
BS3,"[(3, 3, 1)]",[BS3],0.000e+00,0.000
GHZ,"[(3, 3, 3)]",[GME],1.509e-06,0.828
SEP,"[(1, 1, 1)]",[SEP],0.000e+00,0.000
W,"[(3, 3, 3)]",[GME],0.000e+00,0.000


## Table VIII: the 30 experimental states
The paper used an SVD tolerance of $2.5\times10^{-1}$ for experimental and $10^{-10}$ for theoretical tensors. The theoretical column is reproduced exactly; the experimental ranks were computed in Mathematica from the tomography output (`mathematica/correlation_tensor_ranks_3q.nb`) and are tolerance-sensitive — this Python version agrees on 22/30 states at the stated tolerance (see PAPER_MAP.md).

In [3]:
exp = dataset.load_experimental()
th = pd.read_csv('../data/raw/theoretical_targets_3q.csv')
out = []
for (_, e), (_, t) in zip(exp.iterrows(), th.iterrows()):
    re, rt = dataset.row_to_rho(e), dataset.row_to_rho(t)
    out.append({'state': e.state_id, 'class': e['class'], 'F': e.fidelity,
                'R_th': measures.correlation_tensor_ranks(rt, 1e-10), 'R_ex': measures.correlation_tensor_ranks(re, 0.25),
                'tau_th': round(measures.three_tangle_canonical(rt), 1), 'tau_ex': round(measures.three_tangle_canonical(re), 1)})
pd.DataFrame(out)

,state,class,F,R_th,R_ex,tau_th,tau_ex
0,random_sep_2,SEP,0.976,"(1, 1, 1)","(1, 1, 1)",0.0,0.0
1,random_sep_3,SEP,0.959,"(1, 1, 1)","(1, 1, 1)",0.0,0.0
2,random_sep_4,SEP,0.933,"(1, 1, 1)","(2, 2, 1)",0.0,0.0
3,random_sep_5,SEP,0.948,"(1, 1, 1)","(1, 1, 1)",0.0,0.0
4,sep_output,SEP,0.985,"(1, 1, 1)","(1, 1, 1)",0.0,0.0
5,BS1_output,BS1,0.945,"(1, 3, 3)","(1, 3, 3)",0.0,0.0
6,random_bs1_2_output,BS1,0.971,"(1, 3, 3)","(1, 3, 3)",0.0,0.0
7,random_bs1_4_output,BS1,0.981,"(1, 3, 3)","(1, 3, 3)",0.0,0.0
8,random_bs1_5_output,BS1,0.938,"(1, 3, 3)","(1, 1, 1)",0.0,0.0
9,random_bs2_1_output,BS1,0.973,"(1, 3, 3)","(3, 1, 3)",0.0,0.0
